In [1]:
pip install biopython pandas tqdm lxml

   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.7 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.7 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.7 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.7 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.7 MB 360.2 kB/s eta 0:00:07
   ------- -------------------------------- 0.5/2.7 MB 360.2 kB/s eta 0:00:07
   ------- -------------------------------- 0.5/2.7 MB 360.2 kB/s eta 0:00:07
   ------- -------------------------------- 0.5/2.7 MB 360.2 kB/s eta 0:00:07
   ------- -------------------------------- 0.5/2.7 MB 360.2 kB/s eta 0:00:07
   ----------- ---------------------------- 0.8/2.7 MB 301.7 kB/s eta 0:00:07
   ----------- ---------------------------- 0.8/2.7 MB 301.7 kB/s eta 0:00:07
   ----------- ---------------------------- 0.8/2.7 MB 301.7 kB/s eta 0:00:07
   ----------- ----------------

### Pulled PubMed papers (metadata + abstracts)

--> 

In [3]:
from Bio import Entrez
import pandas as pd
from tqdm import tqdm
import time
import os

# ==============================
# CONFIG
# ==============================
Entrez.email = "your_email@gmail.com"   # <-- apna email

QUERY = """
(
    coronary artery disease
    OR atherosclerosis
    OR myocardial infarction
)
AND
(
    risk factors
    OR cholesterol
    OR smoking
    OR hypertension
)
"""

RETMAX = 300
SAVE_PATH = "data/raw/cad_papers.csv"

# ==============================
# CREATE FOLDER
# ==============================
os.makedirs("data/raw", exist_ok=True)

# ==============================
# LOAD EXISTING CSV (resume mode)
# ==============================
if os.path.exists(SAVE_PATH):
    existing_df = pd.read_csv(SAVE_PATH)
    existing_pmids = set(existing_df["pmid"].astype(str))
    papers = existing_df.to_dict("records")
    print(f"Loaded existing papers: {len(existing_pmids)}")
else:
    existing_df = pd.DataFrame()
    existing_pmids = set()
    papers = []
    print("No existing CSV found. Starting fresh.")

# ==============================
# SEARCH PUBMED
# ==============================
print("Searching PubMed...")

handle = Entrez.esearch(
    db="pubmed",
    term=QUERY,
    retmax=RETMAX
)

record = Entrez.read(handle)
pmids = record["IdList"]

print(f"Total papers in search: {len(pmids)}")

# only fetch missing PMIDs
new_pmids = [pmid for pmid in pmids if pmid not in existing_pmids]

print(f"Already downloaded: {len(existing_pmids)}")
print(f"Remaining to fetch: {len(new_pmids)}")

# ==============================
# FETCH ONLY MISSING PAPERS
# ==============================
for pmid in tqdm(new_pmids):
    try:
        handle = Entrez.efetch(
            db="pubmed",
            id=pmid,
            rettype="abstract",
            retmode="xml"
        )

        records = Entrez.read(handle)

        if not records.get("PubmedArticle"):
            print(f"Skipped {pmid}: empty record")
            continue

        article = records["PubmedArticle"][0]
        medline = article["MedlineCitation"]
        article_data = medline["Article"]

        # TITLE
        title = str(article_data.get("ArticleTitle", ""))

        # ABSTRACT
        abstract = ""
        if "Abstract" in article_data:
            abstract = " ".join(
                [str(x) for x in article_data["Abstract"]["AbstractText"]]
            )

        # YEAR
        year = None
        try:
            pubdate = article_data["Journal"]["JournalIssue"]["PubDate"]
            year = pubdate.get("Year", None)
        except:
            pass

        # JOURNAL
        journal = article_data.get("Journal", {}).get("Title", "")

        # AUTHORS
        authors = []
        try:
            for author in article_data.get("AuthorList", []):
                full = f"{author.get('ForeName','')} {author.get('LastName','')}".strip()
                if full:
                    authors.append(full)
        except:
            pass

        # MeSH TERMS
        mesh_terms = []
        try:
            for mesh in medline.get("MeshHeadingList", []):
                mesh_terms.append(str(mesh["DescriptorName"]))
        except:
            pass

        # DOI
        doi = ""
        try:
            for iden in article["PubmedData"]["ArticleIdList"]:
                if iden.attributes.get("IdType") == "doi":
                    doi = str(iden)
                    break
        except:
            pass

        # STUDY TYPE
        study_type = ""
        try:
            study_type = ", ".join(
                [str(x) for x in article_data.get("PublicationTypeList", [])]
            )
        except:
            pass

        papers.append({
            "pmid": pmid,
            "title": title,
            "abstract": abstract,
            "year": year,
            "journal": journal,
            "authors": "; ".join(authors),
            "mesh_terms": "; ".join(mesh_terms),
            "doi": doi,
            "study_type": study_type
        })

        time.sleep(0.3)

    except Exception as e:
        print(f"Skipped {pmid}: {e}")

# ==============================
# FINAL SAVE
# ==============================
df = pd.DataFrame(papers)

# remove duplicates by PMID
df = df.drop_duplicates(subset=["pmid"])

df.to_csv(SAVE_PATH, index=False)

# ==============================
# SUMMARY
# ==============================
print("\nDone!")
print(f"Final dataset shape: {df.shape}")
print(f"Saved to: {SAVE_PATH}")
print(df.head())

No existing CSV found. Starting fresh.
Searching PubMed...
Total papers in search: 300
Already downloaded: 0
Remaining to fetch: 300


 99%|█████████▉| 297/300 [07:35<00:03,  1.00s/it]

Skipped 27809442: empty record


100%|██████████| 300/300 [07:39<00:00,  1.53s/it]


Done!
Final dataset shape: (299, 9)
Saved to: data/raw/cad_papers.csv
       pmid                                              title  \
0  42151716  Natriuretic Peptides as a Biomarker of Cogniti...   
1  42151241  Tanshinone IIA inhibits foam cell formation an...   
2  42151198  Association between lactate-to-albumin ratio a...   
3  42150991  In-Hospital Mortality and Associated Risk Fact...   
4  42150912  [Clinical characteristics and gender differenc...   

                                            abstract  year  \
0  Cognitive decline and dementia are frequent co...  2026   
1  The development of foam cells is crucial in th...  2026   
2  Previous studies have established associations...  2026   
3  To identify independent risk factors for in-ho...  2026   
4  To compare the multidimensional clinical chara...  2026   

                                             journal  \
0  High blood pressure & cardiovascular preventio...   
1                                 Scientific re

In [2]:
from Bio import Entrez
import pandas as pd
import os
from tqdm import tqdm
import time

Entrez.email = "zenilroy34@gmail.com"

INPUT_PATH = "data/raw/cad_papers.csv"
XML_DIR = "data/fulltext_xml"

os.makedirs(XML_DIR, exist_ok=True)

df = pd.read_csv(INPUT_PATH)
pmids = df["pmid"].astype(str).tolist()

downloaded = 0
skipped = 0
already_exists = 0
no_pmc = 0

for pmid in tqdm(pmids):
    try:
        xml_path = os.path.join(XML_DIR, f"{pmid}.xml")

        # Skip already downloaded
        if os.path.exists(xml_path):
            already_exists += 1
            continue

        # PMID -> PMCID
        handle = Entrez.elink(
            dbfrom="pubmed",
            db="pmc",
            id=pmid
        )
        record = Entrez.read(handle)

        linksets = record[0].get("LinkSetDb", [])
        if not linksets:
            no_pmc += 1
            continue

        pmcid = linksets[0]["Link"][0]["Id"]

        # Fetch full XML directly from PMC database
        fetch_handle = Entrez.efetch(
            db="pmc",
            id=pmcid,
            rettype="full",
            retmode="xml"
        )

        xml_data = fetch_handle.read()

        if not xml_data:
            skipped += 1
            continue

        with open(xml_path, "wb") as f:
            f.write(xml_data)

        downloaded += 1
        time.sleep(0.3)

    except Exception as e:
        print(f"Skipped {pmid}: {e}")
        skipped += 1

print("\nDone!")
print("Downloaded XML:", downloaded)
print("Already exists:", already_exists)
print("No PMC full text:", no_pmc)
print("Skipped:", skipped)

 70%|██████▉   | 209/299 [12:15<07:52,  5.25s/it] 

Skipped 42100188: NCBI C++ Exception:
    Error: TXCLIENT(CException::eUnknown) "/pubmed_gen/rbuild/version/20260413/entrez/2.20/src/internal/txclient/TxClient.cpp", line 1100: ncbi::CTxRawClientImpl::readAll() --- Read failed: EOF (the other side has unexpectedly closed connection), peer: 130.14.22.161:8064



 72%|███████▏  | 214/299 [12:44<09:08,  6.45s/it]

Skipped 42099647: NCBI C++ Exception:
    Error: TXCLIENT(CException::eUnknown) "/pubmed_gen/rbuild/version/20260413/entrez/2.20/src/internal/txclient/TxClient.cpp", line 1100: ncbi::CTxRawClientImpl::readAll() --- Read failed: EOF (the other side has unexpectedly closed connection), peer: 130.14.22.161:8064



 88%|████████▊ | 264/299 [15:49<02:06,  3.61s/it]

Skipped 42089099: IncompleteRead(66630 bytes read)


 93%|█████████▎| 278/299 [16:32<00:55,  2.65s/it]

Skipped 42086356: NCBI C++ Exception:
    Error: TXCLIENT(CException::eUnknown) "/pubmed_gen/rbuild/version/20260413/entrez/2.20/src/internal/txclient/TxClient.cpp", line 1100: ncbi::CTxRawClientImpl::readAll() --- Read failed: EOF (the other side has unexpectedly closed connection), peer: 130.14.22.161:8064



100%|██████████| 299/299 [17:35<00:00,  3.53s/it]


Done!
Downloaded XML: 123
Already exists: 0
No PMC full text: 172
Skipped: 4


In [3]:
import os
import json
from bs4 import BeautifulSoup
from tqdm import tqdm

INPUT_DIR = "data/fulltext_xml"
OUTPUT_DIR = "data/parsed_fulltext"

os.makedirs(OUTPUT_DIR, exist_ok=True)

xml_files = [
    f for f in os.listdir(INPUT_DIR)
    if f.endswith(".xml")
]

parsed = 0
skipped = 0

for file in tqdm(xml_files):
    try:
        pmid = file.replace(".xml", "")
        out_path = os.path.join(OUTPUT_DIR, f"{pmid}.json")

        # skip already parsed
        if os.path.exists(out_path):
            continue

        xml_path = os.path.join(INPUT_DIR, file)

        with open(xml_path, "r", encoding="utf-8", errors="ignore") as f:
            soup = BeautifulSoup(f.read(), "xml")

        # ---------------- TITLE ----------------
        title_tag = soup.find("article-title")
        title = title_tag.get_text(" ", strip=True) if title_tag else ""

        # ---------------- ABSTRACT ----------------
        abstract_tag = soup.find("abstract")
        abstract = (
            abstract_tag.get_text(" ", strip=True)
            if abstract_tag else ""
        )

        # ---------------- BODY ----------------
        body_tag = soup.find("body")
        body_text = (
            body_tag.get_text(" ", strip=True)
            if body_tag else ""
        )

        # ---------------- SECTION EXTRACTION ----------------
        sections = {}

        for sec in soup.find_all("sec"):
            sec_title = sec.find("title")
            if sec_title:
                name = sec_title.get_text(" ", strip=True).lower()
                text = sec.get_text(" ", strip=True)

                sections[name] = text

        # Try common sections
        methods = ""
        results = ""
        discussion = ""
        conclusion = ""

        for k, v in sections.items():
            if "method" in k:
                methods += v + "\n"
            elif "result" in k:
                results += v + "\n"
            elif "discussion" in k:
                discussion += v + "\n"
            elif "conclusion" in k:
                conclusion += v + "\n"

        payload = {
            "pmid": pmid,
            "title": title,
            "abstract": abstract,
            "methods": methods.strip(),
            "results": results.strip(),
            "discussion": discussion.strip(),
            "conclusion": conclusion.strip(),
            "body_text": body_text
        }

        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2, ensure_ascii=False)

        parsed += 1

    except Exception as e:
        print(f"Skipped {file}: {e}")
        skipped += 1

print("\nDone!")
print("Parsed:", parsed)
print("Skipped:", skipped)

100%|██████████| 123/123 [00:07<00:00, 16.25it/s]


Done!
Parsed: 123
Skipped: 0


### Now we will create Chunks

In [11]:
import os
import json
from tqdm import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter  # ✅ updated import

INPUT_DIR = "data/parsed_fulltext"
OUTPUT_DIR = "data/chunks"

os.makedirs(OUTPUT_DIR, exist_ok=True)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

all_chunks = []

files = [
    f for f in os.listdir(INPUT_DIR)
    if f.endswith(".json")
]

for file in tqdm(files):
    path = os.path.join(INPUT_DIR, file)

    with open(path, "r", encoding="utf-8") as f:
        doc = json.load(f)

    pmid = doc["pmid"]
    title = doc["title"]

    sections = {
        "abstract": doc.get("abstract", ""),
        "methods": doc.get("methods", ""),
        "results": doc.get("results", ""),
        "discussion": doc.get("discussion", ""),
        "conclusion": doc.get("conclusion", "")
    }

    for sec_name, sec_text in sections.items():
        if not sec_text or len(sec_text.strip()) < 50:
            continue

        chunks = splitter.split_text(sec_text)

        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "chunk_id": f"{pmid}_{sec_name}_{i}",
                "pmid": pmid,
                "title": title,
                "section": sec_name,
                "text": chunk
            })

# Save
out_path = os.path.join(OUTPUT_DIR, "all_chunks.json")

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=2, ensure_ascii=False)

print("Total chunks:", len(all_chunks))
print("Saved:", out_path)


100%|██████████| 123/123 [00:00<00:00, 320.92it/s]


Total chunks: 3467
Saved: data/chunks\all_chunks.json


In [14]:
pip install sentence-transformers faiss-cpu

  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/588.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/588.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/588.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/588.7 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/588.7 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/588.7 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/588.7 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/588.7 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/588.7 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/588.7 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/588.7 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/588.7 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/588.7 kB ? e

In [15]:
import json
import os
import pickle
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

CHUNK_PATH = "data/chunks/all_chunks.json"
VECTOR_DIR = "data/vector_db"

os.makedirs(VECTOR_DIR, exist_ok=True)

# -------------------------
# Load chunks
# -------------------------
with open(CHUNK_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

texts = [x["text"] for x in chunks]

print(f"Loaded chunks: {len(texts)}")

# -------------------------
# Load embedding model
# -------------------------
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

# -------------------------
# Generate embeddings
# -------------------------
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

# normalize for cosine similarity
faiss.normalize_L2(embeddings)

# -------------------------
# Create FAISS index
# -------------------------
dim = embeddings.shape[1]

# Inner product + normalized vectors = cosine similarity
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

# -------------------------
# Save vector DB
# -------------------------
faiss.write_index(
    index,
    os.path.join(VECTOR_DIR, "healthgpt.index")
)

# -------------------------
# Save metadata
# -------------------------
with open(
    os.path.join(VECTOR_DIR, "chunk_metadata.pkl"),
    "wb"
) as f:
    pickle.dump(chunks, f)

print("\nDone!")
print("Embedding shape:", embeddings.shape)
print("Vector count:", index.ntotal)
print("Saved to:", VECTOR_DIR)

Loaded chunks: 3467


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Zenil\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Zenil\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/109 [00:00<?, ?it/s]


Done!
Embedding shape: (3467, 384)
Vector count: 3467
Saved to: data/vector_db


In [16]:
import faiss
import pickle
import json
import numpy as np
from sentence_transformers import SentenceTransformer

# -----------------------------
# LOAD VECTOR DB
# -----------------------------
INDEX_PATH = "data/vector_db/healthgpt.index"
META_PATH = "data/vector_db/chunk_metadata.pkl"

index = faiss.read_index(INDEX_PATH)

with open(META_PATH, "rb") as f:
    metadata = pickle.load(f)

# -----------------------------
# LOAD EMBEDDING MODEL
# -----------------------------
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

# -----------------------------
# SEARCH FUNCTION
# -----------------------------
def search_papers(query, top_k=5):
    # embed query
    q = model.encode(
        [query],
        convert_to_numpy=True
    )

    faiss.normalize_L2(q)

    # search
    scores, indices = index.search(q, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        item = metadata[idx]

        results.append({
            "score": float(score),
            "pmid": item["pmid"],
            "title": item["title"],
            "section": item["section"],
            "text": item["text"][:800]
        })

    return results

# -----------------------------
# TEST QUERY
# -----------------------------
query = "Does smoking increase coronary artery disease risk?"

results = search_papers(query, top_k=5)

for i, r in enumerate(results, 1):
    print("=" * 100)
    print(f"Rank: {i}")
    print(f"Score: {r['score']:.4f}")
    print(f"PMID: {r['pmid']}")
    print(f"Title: {r['title']}")
    print(f"Section: {r['section']}")
    print("Snippet:")
    print(r["text"])
    print()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Rank: 1
Score: 0.7996
PMID: 42094110
Title: Can QRISK-3 and PREVENT Predict Premature Acute Coronary Syndrome in Adults under 40 Years? The Insights from Predicting the Risk of Acute Coronary Syndrome under 40 (PRACS-40) Study
Section: discussion
Snippet:
that while the baseline metabolic parameters and lipid profiles were similar, long-term (30-year) cardiovascular risk estimated by QRISK-3 was significantly higher among smokers. This finding indicates the cumulative vascular damage induced by smoking, not always reflected in short-term risk estimations but evident over extended horizons. Interestingly, smokeless tobacco users had a lower calculated 30-year QRISK-3 score compared to nonusers, which may be due to risk score limitations rather than a true lower risk. Smokeless tobacco use was also linked to shorter sleep duration, potentially compounding cardiovascular risk. These results emphasize that both forms of tobacco use substantially contribute to early coronary artery disease,

In [22]:
import gradio as gr
import faiss
import pickle
from sentence_transformers import SentenceTransformer

# =========================
# LOAD VECTOR DB
# =========================
INDEX_PATH = "data/vector_db/healthgpt.index"
META_PATH = "data/vector_db/chunk_metadata.pkl"

index = faiss.read_index(INDEX_PATH)

with open(META_PATH, "rb") as f:
    metadata = pickle.load(f)

model = SentenceTransformer("BAAI/bge-small-en-v1.5")


# =========================
# RETRIEVER
# =========================
def search_papers(query, top_k=3):
    q = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q)

    scores, indices = index.search(q, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        item = metadata[idx]

        results.append({
            "score": float(score),
            "pmid": item["pmid"],
            "title": item["title"],
            "section": item["section"],
            "text": item["text"][:500]
        })

    return results


# =========================
# CHAT FUNCTION
# =========================
def healthgpt_chat(message, history):
    results = search_papers(message)

    answer = "🩺 **HealthGPT Evidence Summary**\n\n"

    for i, r in enumerate(results, 1):
        answer += (
            f"### {i}. {r['title']}\n"
            f"- PMID: {r['pmid']}\n"
            f"- Section: {r['section']}\n"
            f"- Relevance Score: {r['score']:.4f}\n"
            f"- Snippet: {r['text']}\n\n"
        )

    answer += "\n⚠️ Informational use only — not a diagnosis."

    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": answer})

    return history


# =========================
# UI
# =========================
with gr.Blocks() as demo:
    gr.Markdown("# 🩺 HealthGPT")
    gr.Markdown("### Doctor Research Assistant (PubMed + PMC RAG)")

    chatbot = gr.Chatbot(height=500)  # ✅ removed 'type' argument

    state = gr.State([])

    msg = gr.Textbox(
        placeholder="Ask: Does smoking increase CAD risk?",
        label="Ask HealthGPT"
    )

    clear = gr.Button("Clear Chat")

    msg.submit(
        healthgpt_chat,
        inputs=[msg, state],
        outputs=[chatbot]
    )

    clear.click(lambda: [], outputs=[chatbot])

# ✅ Pass theme to launch instead of Blocks
demo.launch(inline=True, theme=gr.themes.Soft())


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
